# Model 7 - Transfer Learning (MobileNetV2)

**Bonus / different approach.** Every other model learns features from scratch;
this one reuses **MobileNetV2 pre-trained on ImageNet** (base frozen) and trains
only a small `GAP -> Dropout -> Dense` head. The point: borrow a big model's
learned edges/textures/shapes instead of relearning them from ~18k images, and
see how far past the ~74% from-scratch ceiling it gets.

## 1. Setup & imports

Same shared pipeline as the other notebooks; only the model factory imported
differs.

In [1]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "preprocessing", "label_mapping"))
sys.path.append(os.path.join(os.getcwd(), "preprocessing", "data_loader"))
sys.path.append(os.path.join(os.getcwd(), "model"))
sys.path.append(os.path.join(os.getcwd(), "evaluation", "model_metrics"))
sys.path.append(os.path.join(os.getcwd(), "evaluation", "plots"))

import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping

from label_mapping import build_labeled_dataset
from data_loader import build_train_val_test_generators
from transfer_mobilenet import build_transfer_mobilenet
from model_metrics import debug_model, evaluate_model, record_result
from plots import plot_misclassified

## 2. Load & label the dataset

Build the image index (Italian folder -> English label, root-relative paths).

In [2]:
df = build_labeled_dataset()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26179 entries, 0 to 26178
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   image_path  26179 non-null  str  
 1   label_it    26179 non-null  str  
 2   label_en    26179 non-null  str  
dtypes: str(3)
memory usage: 613.7 KB


## 3. Preprocess: split, resize, normalize

Identical preprocessing to every other model (128x128, rescale 1/255, stratified
70/15/15) so the comparison stays fair. The `[0,1] -> [-1,1]` shift MobileNetV2
needs is baked **inside** the model, so this pipeline is untouched.

In [3]:
train_generator, val_generator, test_generator = build_train_val_test_generators(
    df, project_root=os.getcwd(), image_size=(128, 128)
)
train_generator.class_indices

Found 18325 validated image filenames belonging to 10 classes.


Found 3927 validated image filenames belonging to 10 classes.


Found 3927 validated image filenames belonging to 10 classes.


{'butterfly': 0,
 'cat': 1,
 'chicken': 2,
 'cow': 3,
 'dog': 4,
 'elephant': 5,
 'horse': 6,
 'sheep': 7,
 'spider': 8,
 'squirrel': 9}

## 4. Build the model

Frozen **MobileNetV2 (ImageNet)** base + a fresh `GAP -> Dropout -> Dense` head.
Note the summary: ~2.3M total params but only ~13k **trainable** - we only train
the tiny head, which is why this is fast despite the big base.

In [4]:
model = build_transfer_mobilenet(
    input_shape=train_generator.image_shape,
    num_classes=len(train_generator.class_indices),
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 5. Train - or load a saved model

If `models_saved/transfer_mobilenet.keras` exists it's **loaded** (skips retrain);
otherwise it trains with `EarlyStopping` and is saved. Only the head trains
(base frozen), so epochs are cheap. Set `RETRAIN = True` to force a fresh run.

In [5]:
import time
from tensorflow import keras

MODEL_PATH = "models_saved/transfer_mobilenet.keras"
RETRAIN = False

if not RETRAIN and os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    history, train_time_min = None, None
    print(f"Loaded {MODEL_PATH} (skipped training).")
else:
    early_stopping = EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
    start_time = time.time()
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20,
        callbacks=[early_stopping],
    )
    train_time_min = round((time.time() - start_time) / 60, 1)
    print(f"Trained in {train_time_min} min.")

Epoch 1/20


  1/573 ━━━━━━━━━━━━━━━━━━━━ 18:03 2s/step - accuracy: 0.0000e+00 - loss: 3.6819

  3/573 ━━━━━━━━━━━━━━━━━━━━ 27s 48ms/step - accuracy: 0.0521 - loss: 3.4703    

  5/573 ━━━━━━━━━━━━━━━━━━━━ 27s 48ms/step - accuracy: 0.0625 - loss: 3.1816

  7/573 ━━━━━━━━━━━━━━━━━━━━ 27s 48ms/step - accuracy: 0.1027 - loss: 3.1475

...

569/573 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9392 - loss: 0.1894

570/573 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9393 - loss: 0.1891

572/573 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9394 - loss: 0.1890

573/573 ━━━━━━━━━━━━━━━━━━━━ 33s 58ms/step - accuracy: 0.9393 - loss: 0.1892 - val_accuracy: 0.9447 - val_loss: 0.2156


Trained in 4.1 min.


## 6. Training history

Per-epoch accuracy/loss. Transfer learning usually converges in just a few
epochs since the features are already learned.

In [6]:
if history is not None:
    history_df = pd.DataFrame(history.history)
    history_df.index.name = "epoch"
    history_df["acc_gap"] = history_df["accuracy"] - history_df["val_accuracy"]
    display(history_df)
else:
    print("Model was loaded from disk - no training history this run.")

,accuracy,loss,val_accuracy,val_loss,acc_gap
epoch,,,,,
0,0.851460,0.489423,0.932518,0.228689,-0.081059
1,0.924966,0.245954,0.936593,0.220593,-0.011627
2,0.932769,0.222205,0.938885,0.211868,-0.006115
3,0.935880,0.203580,0.940413,0.206152,-0.004533
4,0.935225,0.201626,0.939394,0.214439,-0.004169
5,0.938226,0.190364,0.937357,0.226181,0.000870
6,0.939263,0.189176,0.944742,0.215647,-0.005478


## 7. Evaluate: per-class report + confusion matrix

Score on the validation set; per-class precision/recall/F1 + confusion-matrix
heatmap. Watch whether the weak from-scratch classes (cow, cat) improve.

In [7]:
debug_model(model, val_generator)

  1/123 ━━━━━━━━━━━━━━━━━━━━ 1:04 528ms/step

  2/123 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step   

  3/123 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step

  4/123 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step

  5/123 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step

...

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step


              precision    recall  f1-score   support

   butterfly       0.95      0.97      0.96       317
         cat       0.97      0.91      0.94       250
     chicken       0.96      0.97      0.96       464
         cow       0.88      0.84      0.86       280
         dog       0.95      0.94      0.94       730
    elephant       0.95      0.93      0.94       217
       horse       0.90      0.93      0.91       393
       sheep       0.85      0.89      0.87       273
      spider       0.98      0.98      0.98       723
    squirrel       0.95      0.95      0.95       280

    accuracy                           0.94      3927
   macro avg       0.93      0.93      0.93      3927
weighted avg       0.94      0.94      0.94      3927



C:\Users\GBT H610M-\Documents\ironHack\task\project-2\evaluation\model_metrics\model_metrics.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,butterfly,cat,chicken,cow,dog,elephant,horse,sheep,spider,squirrel
butterfly,307,0,1,0,1,0,0,0,8,0
cat,1,227,3,0,12,0,1,0,0,6
chicken,2,0,449,2,4,0,2,2,1,2
cow,0,0,2,235,5,2,18,16,1,1
dog,2,4,7,4,689,3,11,7,1,2
elephant,0,0,0,1,6,202,2,5,1,0
horse,2,0,2,9,7,0,365,7,0,1
sheep,0,1,2,15,1,5,6,242,0,1
spider,8,1,1,0,1,0,0,1,710,1
squirrel,1,2,1,0,3,1,0,4,1,267


## 8. Misclassified images

Grid of validation images the model got wrong (true -> pred).

In [8]:
plot_misclassified(model, val_generator)

  1/123 ━━━━━━━━━━━━━━━━━━━━ 10s 87ms/step

  2/123 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step 

  3/123 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step

  4/123 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step

  5/123 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step

...

119/123 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

121/123 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step


C:\Users\GBT H610M-\Documents\ironHack\task\project-2\evaluation\plots\plots.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Record result to the tracking sheet

Upsert this model's validation row into `model_tracking.csv`.

In [9]:
learning_rate = round(float(model.optimizer.learning_rate.numpy()), 6)

row = evaluate_model(model, val_generator, "transfer_mobilenet")
row.update({
    "architecture": "MobileNetV2 (ImageNet, frozen) -> GlobalAveragePooling -> Dropout(0.5)",
    "learning_rate": learning_rate,
    "train_time_min": train_time_min,
    "notes": "Transfer learning (val metrics). Frozen ImageNet base, only head trained. Bonus / different approach.",
})
record_result(row)

  1/123 ━━━━━━━━━━━━━━━━━━━━ 9s 81ms/step

  3/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

  5/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

  7/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

  9/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

...

119/123 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step

121/123 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step


,model,architecture,learning_rate,train_time_min,accuracy,precision,recall,f1,notes
0,baseline_cnn,3 conv blocks (6 Conv) -> Flatten -> Dropout(0.5),0.001,15.6,0.6988,0.6875,0.6569,0.6677,Baseline (val metrics). Overfits (~13% train-v...
1,gap_cnn,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,32.3,0.7403,0.7321,0.7111,0.7136,Flatten -> GAP (val metrics). Less overfit (~8...
2,gap_bn_cnn,3 conv blocks (Conv-BN-ReLU) -> GlobalAverageP...,0.001,62.6,0.6924,0.7526,0.6448,0.6593,GAP + BatchNorm (val metrics). Test if BN beat...
3,gap_aug,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,47.3,0.6677,0.6511,0.6433,0.6300,GAP + train augmentation (flip/rotate/shift/zo...
4,gap_aug2,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,54.7,0.7179,0.7065,0.7053,0.6895,Augmentation retry: dropout 0.3 + 60 epochs. T...
5,gap_deep,"4 conv blocks (8 Conv, +256) -> GlobalAverageP...",0.001,14.6,0.1859,0.0186,0.1000,0.0314,FAILED to train: plain deep CNN collapsed to ~...
6,transfer_mobilenet,"MobileNetV2 (ImageNet, frozen) -> GlobalAverag...",0.001,4.1,0.9404,0.9338,0.9309,0.9321,Transfer learning (val metrics). Frozen ImageN...


## 10. Save the model

Save only when freshly trained (a loaded model is already on disk).

In [10]:
if history is not None:
    os.makedirs("models_saved", exist_ok=True)
    model.save(MODEL_PATH)
    print(f"Saved {MODEL_PATH}")
else:
    print(f"Using existing {MODEL_PATH}")

Saved models_saved/transfer_mobilenet.keras


## 11. Compare all models

Bar chart of validation accuracy across every recorded model.

In [11]:
from model_metrics import plot_accuracy_comparison

plot_accuracy_comparison()

C:\Users\GBT H610M-\Documents\ironHack\task\project-2\evaluation\model_metrics\model_metrics.py:109: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,architecture,learning_rate,train_time_min,accuracy,precision,recall,f1,notes
5,gap_deep,"4 conv blocks (8 Conv, +256) -> GlobalAverageP...",0.001,14.6,0.1859,0.0186,0.1000,0.0314,FAILED to train: plain deep CNN collapsed to ~...
3,gap_aug,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,47.3,0.6677,0.6511,0.6433,0.6300,GAP + train augmentation (flip/rotate/shift/zo...
2,gap_bn_cnn,3 conv blocks (Conv-BN-ReLU) -> GlobalAverageP...,0.001,62.6,0.6924,0.7526,0.6448,0.6593,GAP + BatchNorm (val metrics). Test if BN beat...
0,baseline_cnn,3 conv blocks (6 Conv) -> Flatten -> Dropout(0.5),0.001,15.6,0.6988,0.6875,0.6569,0.6677,Baseline (val metrics). Overfits (~13% train-v...
4,gap_aug2,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,54.7,0.7179,0.7065,0.7053,0.6895,Augmentation retry: dropout 0.3 + 60 epochs. T...
1,gap_cnn,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,32.3,0.7403,0.7321,0.7111,0.7136,Flatten -> GAP (val metrics). Less overfit (~8...
6,transfer_mobilenet,"MobileNetV2 (ImageNet, frozen) -> GlobalAverag...",0.001,4.1,0.9404,0.9338,0.9309,0.9321,Transfer learning (val metrics). Frozen ImageN...
